In [ ]:
import synapseclient

import pandas as pd
import great_expectations as gx

from agoradatatools.gx import GreatExpectationsRunner

context = gx.get_context(project_root_dir='../src/agoradatatools/great_expectations')

from expectations.expect_column_values_to_have_list_length_in_range import ExpectColumnValuesToHaveListLengthInRange
from expectations.expect_column_values_to_have_list_members_of_type import ExpectColumnValuesToHaveListMembersOfType

# Create Expectation Suite for Nominated Drugs Data

## Get Example Data File

In [ ]:
syn = synapseclient.Synapse()
syn.login()

In [ ]:
nominated_drugs_data_file = syn.get("syn73695286").path

## Create Validator Object on Data File

In [ ]:
df = pd.read_json(nominated_drugs_data_file)
nested_columns = []
df = GreatExpectationsRunner.convert_nested_columns_to_json(df, nested_columns)
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "nominated_drugs"

## Add Expectations to Validator Object For Each Column

In [ ]:
# common_name
validator.expect_column_values_to_be_of_type("common_name", "str")
validator.expect_column_values_to_not_be_null("common_name")
validator.expect_column_value_lengths_to_be_between("common_name", min_value=1, max_value=250)

In [ ]:
# chembl_id
validator.expect_column_values_to_be_of_type("chembl_id", "str")
validator.expect_column_values_to_not_be_null("chembl_id")
# chembl_id is not unique: combination drugs can share a ChEMBL ID across rows
validator.expect_column_values_to_match_regex("chembl_id", "^CHEMBL\\d+$")

In [ ]:
# total_nominations
validator.expect_column_values_to_be_of_type("total_nominations", "int")
validator.expect_column_values_to_not_be_null("total_nominations")
validator.expect_column_values_to_be_between("total_nominations", min_value=1)

In [ ]:
# combined_with (nullable; only populated for drug combinations)
validator.expect_column_values_to_be_of_type("combined_with", "str")
validator.expect_column_value_lengths_to_be_between(
    "combined_with",
    min_value=1,
    max_value=250,
    row_condition="combined_with.notnull()",
    condition_parser="pandas",
)

In [ ]:
# initial_nomination (year of earliest nomination)
validator.expect_column_values_to_be_of_type("initial_nomination", "int")
validator.expect_column_values_to_not_be_null("initial_nomination")
validator.expect_column_values_to_be_between("initial_nomination", min_value=1900, max_value=2100)

In [ ]:
# principal_investigators
validator.expect_column_values_to_be_of_type("principal_investigators", "list")
validator.expect_column_values_to_not_be_null("principal_investigators")
validator.expect_column_values_to_have_list_length_in_range(column="principal_investigators", list_length_range=[1, 100])
validator.expect_column_values_to_have_list_members_of_type(column="principal_investigators", member_type="str")

In [ ]:
# programs
validator.expect_column_values_to_be_of_type("programs", "list")
validator.expect_column_values_to_not_be_null("programs")
validator.expect_column_values_to_have_list_length_in_range(column="programs", list_length_range=[1, 100])
validator.expect_column_values_to_have_list_members_of_type(column="programs", member_type="str")

In [ ]:
# modality (nullable; populated from OpenTargets metadata)
validator.expect_column_values_to_be_of_type("modality", "str")
validator.expect_column_values_to_be_in_set(
    "modality",
    ["Small molecule", "Protein"],
    row_condition="modality.notnull()",
    condition_parser="pandas",
)

In [ ]:
# year_of_first_approval (nullable; loaded as float because of nulls)
validator.expect_column_values_to_be_of_type("year_of_first_approval", "float")
validator.expect_column_values_to_be_between(
    "year_of_first_approval",
    min_value=1900,
    max_value=2100,
    row_condition="year_of_first_approval.notnull()",
    condition_parser="pandas",
)

In [ ]:
# maximum_clinical_trial_phase (nullable)
validator.expect_column_values_to_be_of_type("maximum_clinical_trial_phase", "str")
validator.expect_column_values_to_be_in_set(
    "maximum_clinical_trial_phase",
    ["Phase I", "Phase II", "Phase III", "Phase IV", "Preclinical", "Unknown"],
    row_condition="maximum_clinical_trial_phase.notnull()",
    condition_parser="pandas",
)

## Save Expectation Suite

In [ ]:
validator.save_expectation_suite(discard_failed_expectations=False)

## Create Checkpoint and View Results

In [ ]:
checkpoint = context.add_or_update_checkpoint(
    name="agora-test-checkpoint",
    validator=validator,
)
checkpoint_result = checkpoint.run()
context.view_validation_result(checkpoint_result)

## Build Data Docs - Click on Expectation Suite to View All Expectations

In [ ]:
context.build_data_docs()
context.open_data_docs()